# Step 4 — Calculate vegetation indices

Computes NDVI, NDWI and SAVI from the red, green and near-infrared bands, on the
common analysis grid.

| | |
|---|---|
| Six-phase position | Scientific computation |
| W1 Algae Bloom counterpart | `calculate-band` (band algebra) |
| Outputs | `ndvi_file`, `ndwi_file`, `savi_file` (float32 GeoTIFF) |

Bands are recognised by the prefix of their file name (`red…`, `green…`, `nir…`),
as written by the previous steps.

In [ ]:
from pathlib import Path

import numpy as np
import rasterio

In [ ]:
# CWL type annotations (removed by ipython2cwl in the generated tool)
from typing import List, Optional

from ipython2cwl.iotypes import (
    CWLDirectoryPathOutput,
    CWLFilePathInput,
    CWLFilePathOutput,
    CWLFloatInput,
    CWLIntInput,
    CWLMetadata,
    CWLNamespaces,
    CWLRequirement,
    CWLStringInput,
)

In [ ]:
cwl_requirements: CWLRequirement = {
    "ResourceRequirement": {"coresMin": 1, "ramMin": 1024},
}

In [ ]:
cwl_metadata: CWLMetadata = {
    "s:softwareVersion": "0.1.0",
    "s:keywords": ["ospd", "mangrove", "ndvi", "ndwi", "savi"],
    "s:author": [{"class": "s:Person", "s:name": "Cameron Sajedi"}],
    "s:contributor": [
        {"class": "s:Person", "s:name": "Gérald Fenoy", "s:affiliation": "GeoLabs"}
    ],
    "s:codeRepository": "https://github.com/starling-foundries/KindGrove",
    "s:license": "https://spdx.org/licenses/CC-BY-NC-SA-4.0",
    "s:description": "Compute NDVI, NDWI and SAVI from Sentinel-2 red, green and NIR bands",
}

In [ ]:
cwl_namespaces: CWLNamespaces = {
    "s": "https://schema.org/",
}

## Inputs

In [ ]:
band_files: List[CWLFilePathInput] = ["red_4326.tif", "green_4326.tif", "nir_4326.tif"]

## Read the bands

In [ ]:
bands = {}
profile = None
for path in band_files:
    name = Path(path).stem.split("_")[0]
    with rasterio.open(path) as src:
        bands[name] = src.read(1).astype("float64")
        if profile is None:
            profile = src.profile.copy()
        elif (src.transform, src.width, src.height) != (profile["transform"], profile["width"], profile["height"]):
            raise ValueError(f"{path} is not on the same grid as {band_files[0]}")
missing = {"red", "green", "nir"} - set(bands)
if missing:
    raise ValueError(f"Missing bands {sorted(missing)} in {band_files}")
red, green, nir = bands["red"], bands["green"], bands["nir"]

## Indices

In [ ]:
ndvi = (nir - red) / (nir + red + 1e-8)
ndwi = (green - nir) / (green + nir + 1e-8)
savi = ((nir - red) / (nir + red + 0.5)) * 1.5
print(f"NDVI range: {np.nanmin(ndvi):.3f} to {np.nanmax(ndvi):.3f}")

In [ ]:
profile.update(dtype="float32", nodata=np.nan, compress="deflate")


def save(path, data, description):
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(data.astype("float32"), 1)
        dst.set_band_description(1, description)
    print(f"Saved: {path}")


ndvi_file: CWLFilePathOutput = "ndvi.tif"
save(ndvi_file, ndvi, "NDVI")
ndwi_file: CWLFilePathOutput = "ndwi.tif"
save(ndwi_file, ndwi, "NDWI")
savi_file: CWLFilePathOutput = "savi.tif"
save(savi_file, savi, "SAVI")